In [ ]:
# ar/data-analysis/normal/07-groupby-basics
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("titanic.csv", ())


## نمط التقسيم-والتطبيق-والدمج

يعتبر GroupBy أحد أقوى ميزات pandas. يتبع نمطًا من ثلاث خطوات:

1. **التقسيم** — قسّم إطار البيانات إلى مجموعات بناءً على عمود واحد أو أكثر
2. **التطبيق** — احسب دالة على كل مجموعة بشكل مستقل
3. **الدمج** — ادمج النتائج مرة أخرى في إطار بيانات واحد


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")


## التجميع حسب عمود واحد


In [ ]:
# Average survival rate by passenger class
print(df.groupby("Pclass")["Survived"].mean())


المخرجات:


In [ ]:
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


كان معدل نجاة ركاب الدرجة الأولى 63%، مقارنة بـ 24% للدرجة الثالثة. كشف groupby عن فجوة طبقية صارخة في ثوانٍ.

**ما يحدث خطوة بخطوة:**


In [ ]:
# This is conceptually what groupby does:
for pclass, group_df in df.groupby("Pclass"):
    print(f"Class {pclass}: {group_df['Survived'].mean():.3f}")


## التجميع حسب أعمدة متعددة


In [ ]:
# Survival rate by class and sex
print(df.groupby(["Pclass", "Sex"])["Survived"].mean())


المخرجات:


In [ ]:
Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64


استخدم `unstack()` لجعل هذا أسهل قراءة:


In [ ]:
print(df.groupby(["Pclass", "Sex"])["Survived"].mean().unstack())


## أساليب التجميع

يدعم GroupBy جميع عمليات التجميع القياسية:


In [ ]:
# Mean fare by class
print(df.groupby("Pclass")["Fare"].mean())

# Total fare collected per class
print(df.groupby("Pclass")["Fare"].sum())

# Count of passengers per class
print(df.groupby("Pclass")["PassengerId"].count())


## عمليات تجميع متعددة باستخدام agg()

تطبق طريقة `agg()` دوال متعددة دفعة واحدة:


In [ ]:
print(df.groupby("Pclass")["Fare"].agg(["mean", "median", "min", "max", "count"]))


المخرجات:


In [ ]:
              mean  median     min       max  count
Pclass                                             
1        84.154687  60.287  0.0000  512.3292    216
2        20.662183  19.575  0.0000   73.5000    184
3        13.675550   8.050  0.0000   56.4958    491


**عمليات تجميع مختلفة لكل عمود:**


In [ ]:
print(df.groupby("Pclass").agg({
    "Survived": "mean",
    "Fare": ["mean", "max"],
    "Age": "median",
    "Name": "count"
}))


## تجميع جميع الأعمدة الرقمية


In [ ]:
# Quick summary of all numeric columns per group
print(df.groupby("Pclass").mean(numeric_only=True))


## GroupBy مع الفلاتر

بعد التجميع، يمكنك تصفية مجموعات كاملة:


In [ ]:
# Keep only groups with more than 50 passengers
large_groups = df.groupby("Pclass").filter(lambda x: len(x) > 50)
print(large_groups["Pclass"].value_counts())


## جرّب بنفسك

باستخدام مجموعة بيانات تيتانيك، احسب:
1. متوسط الأجرة لكل ميناء إقلاع
2. معدل النجاة لكل توليفة من الجنس وميناء الإقلاع
3. إحصاءات العمر (المتوسط والوسيط والأدنى والأقصى) لكل درجة ركاب


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

print("Average fare by port:")
print(df.groupby("Embarked")["Fare"].mean())

print("\nSurvival by sex and port:")
print(df.groupby(["Sex", "Embarked"])["Survived"].mean().unstack())

print("\nAge stats by class:")
print(df.groupby("Pclass")["Age"].agg(["mean", "median", "min", "max"]))


## خلاصات رئيسية

- يتبع GroupBy نمط التقسيم-والتطبيق-والدمج: قسّم البيانات، طبق دالة، ادمج النتائج
- جمّع بعمود واحد للملخصات البسيطة، وبأعمدة متعددة لتحليل أعمق
- يتيح لك `agg()` حساب إحصاءات متعددة مرة واحدة، لكل عمود إذا لزم الأمر
- يكشف GroupBy أنماطًا غير مرئية في البيانات الخام

## تحدي التطبيق

من مجموعة بيانات تيتانيك، احسب معدل النجاة لكل توليفة من Pclass و Sex وما إذا كان الراكب يسافر بمفرده (SibSp + Parch == 0). أي مجموعة حققت أعلى معدل نجاة؟ وأيها الأدنى؟


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
